# Mini Workshop — ONNX

> *PyTorch is the language you wrote your model in. ONNX is the language your model travels in.*

In this Mini, we'll see SceneSeg (an Autoware perception model) running **two ways**:
- **As a PyTorch model** (`.pt`)
- **As an ONNX model** (`.onnx`)

Then you'll do the **conversion yourself** — taking the PyTorch model and exporting it to ONNX, and verifying the output matches.

This is the export pattern you'll use on every model you ever ship. The big Workshop applies it on a full deployment pipeline (PyTorch → ONNX → TensorRT). Here we just learn the move.


## Setup


In [ ]:
!pip install onnxruntime-gpu gdown onnxscript -q


In [ ]:
import torch
import torchvision.transforms as T
import numpy as np
import cv2
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import onnxruntime as ort
import glob, os

print(f"PyTorch {torch.__version__} | ORT {ort.__version__}")


## Download — SceneSeg model & a Waymo frame

We use the same SceneSeg model from Autoware that the big Workshop uses, so that everything connects.
- `SceneSeg_traced.pt` — the PyTorch (TorchScript) model
- `SceneSeg_FP32.onnx` — the same model already exported by Autoware

Plus a few Waymo driving frames to run inference on.


In [ ]:
!mkdir -p /content/data /content/models

# SceneSeg — PyTorch (traced) + pre-exported ONNX
!gdown '1G2pKrjEGLGY1ouQdNPh11N-5LlmDI7ES' -O /content/models/SceneSeg_traced.pt
!gdown -O /content/models/ 'https://docs.google.com/uc?export=download&id=1l-dniunvYyFKvLD7k16Png3AsVTuMl9f'

# Waymo driving frames
!wget -qq https://optical-flow-data.s3.eu-west-3.amazonaws.com/waymo_images.zip -O /content/data/waymo.zip
!unzip -qq /content/data/waymo.zip -d /content/data/

for f in sorted(glob.glob('/content/models/*')):
    print(f"  {Path(f).name:<32} {os.path.getsize(f)/1e6:.1f} MB")


In [ ]:
# What input shape does this model expect? Probe the ONNX file.
_sess = ort.InferenceSession('/content/models/SceneSeg_FP32.onnx', providers=['CPUExecutionProvider'])
_inp  = _sess.get_inputs()[0]
H = int(_inp.shape[2])
W = int(_inp.shape[3])
print(f"Model expects: {H} × {W}")


## Load a Waymo frame

We'll run the same image through both runtimes and compare.


In [ ]:
frames = sorted(glob.glob('/content/data/night/front_images_night/*.jpg'))
frame_path = frames[100]

frame_bgr = cv2.imread(frame_path)
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

preprocess = T.Compose([
    T.Resize((H, W)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

x    = preprocess(Image.fromarray(frame_rgb)).unsqueeze(0)  # PyTorch tensor
x_np = x.numpy()                                            # numpy for ONNX Runtime

plt.imshow(cv2.resize(frame_rgb, (W, H))); plt.axis('off')
plt.title('Input: Waymo driving frame'); plt.show()


In [ ]:
COLORS = np.array([
    [240,  40,  40],   # 0 background
    [180,  60, 200],   # 1 foreground (cars / pedestrians)
    [ 80, 200,  80],   # 2 drivable road
], dtype=np.uint8)


---
## Part 1 — Run SceneSeg as PyTorch

`torch.jit.load` reads the TorchScript file directly — no class definition needed.


In [ ]:
model = torch.jit.load('/content/models/SceneSeg_traced.pt', map_location='cpu')
model.eval()

with torch.no_grad():
    pt_out = model(x).numpy()

pt_map = np.argmax(pt_out[0], axis=0)
print(f"PyTorch output shape: {pt_out.shape}")

# Visual: input | segmentation | overlay
input_img = cv2.resize(frame_rgb, (W, H))
overlay   = cv2.addWeighted(input_img, 0.5, COLORS[pt_map], 0.5, 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].imshow(input_img);     axes[0].set_title('Input frame');           axes[0].axis('off')
axes[1].imshow(COLORS[pt_map]); axes[1].set_title('PyTorch — segmentation'); axes[1].axis('off')
axes[2].imshow(overlay);       axes[2].set_title('PyTorch — overlay');     axes[2].axis('off')
plt.tight_layout(); plt.show()


---
## Part 2 — Run SceneSeg as ONNX

Same model, same input — different runtime. The ONNX file is just a description of the graph; ONNX Runtime executes it.


In [ ]:
sess = ort.InferenceSession(
    '/content/models/SceneSeg_FP32.onnx',
    providers=['CPUExecutionProvider'],   # CUDAExecutionProvider also works on Colab GPU
)
inp_meta = sess.get_inputs()[0]
out_meta = sess.get_outputs()[0]
print(f"ONNX input  : {inp_meta.name} {inp_meta.shape}")
print(f"ONNX output : {out_meta.name} {out_meta.shape}")

ort_out = sess.run(None, {inp_meta.name: x_np})[0]
ort_map = np.argmax(ort_out[0], axis=0)
print(f"ONNX output shape: {ort_out.shape}")

# Visual: input | PyTorch overlay | ONNX overlay
input_img    = cv2.resize(frame_rgb, (W, H))
pt_overlay   = cv2.addWeighted(input_img, 0.5, COLORS[pt_map],  0.5, 0)
ort_overlay  = cv2.addWeighted(input_img, 0.5, COLORS[ort_map], 0.5, 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].imshow(input_img);    axes[0].set_title('Input frame');          axes[0].axis('off')
axes[1].imshow(pt_overlay);   axes[1].set_title('PyTorch overlay');      axes[1].axis('off')
axes[2].imshow(ort_overlay);  axes[2].set_title('ONNX Runtime overlay'); axes[2].axis('off')
plt.tight_layout(); plt.show()

print(f"\nMax abs diff PyTorch vs ONNX: {np.abs(pt_out - ort_out).max():.2e}")


---
## Part 3 — Export Your Own ONNX

Now you'll do what Autoware did — take the PyTorch model and export it to ONNX yourself.

`torch.onnx.export()` needs four things:
1. **The model** — must be in `eval()` mode
2. **A dummy input** — a tensor with the shape the model expects (it traces the graph by running this through)
3. **An output filename**
4. **An opset version** — which set of ONNX operators to use. Opset 17 is a safe modern choice for TensorRT.

Optional but recommended: `input_names` and `output_names` to give the graph readable names.


In [ ]:
dummy = torch.randn(1, 3, H, W)

# dynamo=False -> use the legacy ONNX exporter, which handles TorchScript models.
# (The new dynamo-based path doesn't support ScriptModules yet.)
torch.onnx.export(
    model, dummy,
    '/content/models/SceneSeg_MINE.onnx',
    opset_version=17,
    input_names=['image'],
    output_names=['segmentation'],
    dynamo=False,
)

print(f"Exported: {os.path.getsize('/content/models/SceneSeg_MINE.onnx') / 1e6:.1f} MB")


### Verify your ONNX matches PyTorch numerically

The whole point of ONNX export is that the math doesn't change. Same input → same output (down to floating-point noise).


In [ ]:
sess_mine = ort.InferenceSession(
    '/content/models/SceneSeg_MINE.onnx',
    providers=['CPUExecutionProvider'],
)
mine_out = sess_mine.run(None, {'image': x_np})[0]

max_diff  = np.abs(pt_out - mine_out).max()
mean_diff = np.abs(pt_out - mine_out).mean()
print(f"Max  abs diff PyTorch vs your ONNX: {max_diff:.2e}")
print(f"Mean abs diff PyTorch vs your ONNX: {mean_diff:.2e}")

# np.allclose tolerates tiny FP differences from kernel reordering
match = np.allclose(pt_out, mine_out, atol=1e-4)
print(f"\nNumerical match: {'✅ YES' if match else '❌ NO — something went wrong'}")


### Bonus — Dynamic batch size

By default, the export bakes in batch=1 forever. Often you want batches to be flexible (e.g., process 4 frames at once on a stronger GPU). That's what `dynamic_axes` is for.


In [ ]:
torch.onnx.export(
    model, dummy,
    '/content/models/SceneSeg_DYNAMIC.onnx',
    opset_version=17,
    input_names=['image'],
    output_names=['segmentation'],
    dynamic_axes={
        'image':        {0: 'batch'},
        'segmentation': {0: 'batch'},
    },
    dynamo=False,
)

# Now run with batch=2
sess_dyn = ort.InferenceSession('/content/models/SceneSeg_DYNAMIC.onnx', providers=['CPUExecutionProvider'])
batch2 = np.concatenate([x_np, x_np], axis=0)   # (2, 3, H, W)
out2   = sess_dyn.run(None, {'image': batch2})[0]
print(f"Batch 2 output shape: {out2.shape}")


---
## What's next

You exported a PyTorch model to ONNX, ran it with ONNX Runtime, and verified the math survived.

In the Workshop (`Model_Deployment.ipynb`), we take this same ONNX and:
1. Compile it into a **TensorRT engine** for the GPU
2. Calibrate it to **INT8** using real driving frames
3. Benchmark all 5 runtimes side by side and produce a comparison video

That's the full deployment story — see you there.
